# 05 — Inference & Classification

Apply a trained Vision Transformer (or baseline) model to new, unseen 244×244×64 AlphaEarth embedding tiles
and predict a 6-class mangrove coverage label for each one.

Pipeline step: Classification (step 5 from the paper workflow).

Inputs:
- Trained model checkpoint (`.ckpt` file from PyTorch Lightning)
- Directory of `.npz` embedding files to classify

Outputs:
- CSV with predicted classes & confidence scores per tile

In [ ]:
import torch
import numpy as np
import pandas as pd
import glob
import os
from pathlib import Path
from tqdm.notebook import tqdm

from model.training.models import ViTClassifier, CNNClassifier, Classifier
from model.training.modules import LitModule

## 1. Configuration

Set the paths to your trained checkpoint and the embedding tiles to classify.

In [ ]:
CHECKPOINT_PATH = '../../output/checkpoints/best_model.ckpt'  # Path to trained .ckpt
EMBEDDING_DIR   = '../../output/dataset/'                      # Directory with .npz files
OUTPUT_CSV      = '../../output/predictions/predictions.csv'  # Where to save results
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model architecture (must match the checkpoint)
MODEL_CLASS = 'ViTClassifier'  # or 'CNNClassifier', 'Classifier'

CATEGORY_NAMES = ['1-20%', '21-40%', '41-60%', '61-80%', '81-100%']

print(f'Device: {DEVICE}')

## 2. Load Model

In [ ]:
model_classes = {
    'ViTClassifier': ViTClassifier,
    'CNNClassifier': CNNClassifier,
    'Classifier': Classifier,
}

net = model_classes[MODEL_CLASS](in_channels=64, out_features=5)

# Load from Lightning checkpoint
if os.path.exists(CHECKPOINT_PATH):
    lit = LitModule.load_from_checkpoint(CHECKPOINT_PATH, net=net)
    model = lit.net
    print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
else:
    model = net
    print(f'WARNING: Checkpoint not found at {CHECKPOINT_PATH}, using untrained model.')

model = model.to(DEVICE).eval()
print(f'Model parameters: {model.num_params():,}')

## 3. Run Inference on All Tiles

In [ ]:
npz_files = sorted(glob.glob(os.path.join(EMBEDDING_DIR, '*.npz')))
print(f'Found {len(npz_files)} embedding tiles to classify.')

results = []

with torch.no_grad():
    for fpath in tqdm(npz_files, desc='Classifying'):
        data = np.load(fpath)
        key = 'embeddings' if 'embeddings' in data else list(data.keys())[0]
        emb = torch.from_numpy(data[key].astype(np.float32)).unsqueeze(0).to(DEVICE)  # (1, 64, H, W)

        logits = model(emb)                            # (1, num_classes)
        probs = torch.softmax(logits, dim=1)[0]        # (num_classes,)
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()

        results.append({
            'file': Path(fpath).name,
            'predicted_class': pred_class,
            'predicted_label': CATEGORY_NAMES[pred_class],
            'confidence': round(confidence, 4),
            **{f'prob_{name}': round(probs[i].item(), 4)
               for i, name in enumerate(CATEGORY_NAMES)}
        })

df = pd.DataFrame(results)
print(df.head(10))

## 4. Save Predictions

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df)} predictions to {OUTPUT_CSV}')

## 5. Prediction Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
df['predicted_label'].value_counts().sort_index().plot.bar(
    ax=axes[0], color='teal', edgecolor='k')
axes[0].set_title('Predicted Coverage Distribution')
axes[0].set_ylabel('Count')

# Confidence histogram
axes[1].hist(df['confidence'], bins=20, color='coral', edgecolor='k')
axes[1].set_title('Prediction Confidence')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()